## 电影域元数据预处理（2023版）
只保留同时有 title + images 的物品，优先取 hi_res 图片

In [1]:
import json, gzip, csv, os

META_PATH = "./meta_Movies_and_TV.jsonl.gz"
SAVE_DIR = "./processed/"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# 从images数组中挑最佳图片URL: hi_res > large > 1920w > 720w > 480w > thumb
def pick_best_image(images):
    if not images:
        return ''
    img = images[0]
    for key in ['hi_res', 'large', '1920w', '1440w', '1080w', '720w', '480w', '360w', 'thumb']:
        url = img.get(key, '')
        if url:
            return url
    return ''

total = 0
kept = 0
no_title = 0
no_image = 0

with gzip.open(META_PATH, 'rt', encoding='utf-8') as f_in, \
     open(f'{SAVE_DIR}/movie_meta_clean.csv', 'w', encoding='utf-8', newline='') as f_out:
    
    writer = csv.DictWriter(f_out, fieldnames=['parent_asin', 'title', 'description', 'imUrl'])
    writer.writeheader()
    
    for line in f_in:
        total += 1
        try:
            obj = json.loads(line.strip())
        except:
            continue
        
        title = (obj.get('title') or '').strip()
        images = obj.get('images') or []
        
        if not title:
            no_title += 1
            continue
        if not images:
            no_image += 1
            continue
        
        kept += 1
        
        desc = obj.get('description') or []
        if isinstance(desc, list):
            desc = ' '.join(d for d in desc if d and d.strip())
        desc = (desc or '').strip()
        
        writer.writerow({
            'parent_asin': obj.get('parent_asin', ''),
            'title': title,
            'description': desc,
            'imUrl': pick_best_image(images)
        })
        
        if total % 100000 == 0:
            print(f'  已处理 {total} 条, 保留 {kept} ...')

print(f'总行数: {total}')
print(f'保留: {kept} ({kept/total*100:.1f}%)')
print(f'无title: {no_title}')
print(f'无images: {no_image}')
print(f'输出: {SAVE_DIR}/movie_meta_clean.csv')

  已处理 100000 条, 保留 87395 ...
  已处理 200000 条, 保留 140028 ...
  已处理 400000 条, 保留 245260 ...
  已处理 500000 条, 保留 297748 ...
  已处理 700000 条, 保留 405772 ...
总行数: 748224
保留: 434061 (58.0%)
无title: 314002
无images: 161
输出: ./processed//movie_meta_clean.csv


In [3]:
import pandas as pd
df = pd.read_csv(f'{SAVE_DIR}/movie_meta_clean.csv')
print(f'行数: {len(df)}')
print(f'有描述: {(df["description"]!="").sum()} ({(df["description"]!="").sum()/len(df)*100:.1f}%)')
print(f'\n图片URL样本:')
for url in df['imUrl'].head(3):
    print(f'  {url[:120]}...')
print(f'\n前5条:')
df.head()

行数: 434061
有描述: 434061 (100.0%)

图片URL样本:
  https://images-na.ssl-images-amazon.com/images/S/pv-target-images/8251ee0b9f888d262cd817a5f1aee0b29ffed56a4535af898b8272...
  https://images-na.ssl-images-amazon.com/images/S/pv-target-images/79cd935f5fc3d27d45cb244c2a821ecaaef4624741111680d14009...
  https://m.media-amazon.com/images/I/61rjEQ2h9PL._SL1000_.jpg...

前5条:


,parent_asin,title,description,imUrl
0,B00ABWKL3I,Glee,"Entering its fourth season, this year the memb...",https://images-na.ssl-images-amazon.com/images...
1,B09WDLJ4HP,One Perfect Wedding,With her book tour in two weeks and his expand...,https://images-na.ssl-images-amazon.com/images...
2,B00AHN851G,How to Make Animatronic Characters - Organic M...,Product Description In PART TWO of this incred...,https://m.media-amazon.com/images/I/61rjEQ2h9P...
3,B01G9ILXXE,Ode to Joy: Beethoven's Symphony No. 9,This special Ode to Joy: Beethoven's Symphony ...,https://m.media-amazon.com/images/G/01/digital...
4,B009SIYXDA,Ben 10: Alien Force (Classic),Itâ€™s hero time again for Ben as he saves the...,https://images-na.ssl-images-amazon.com/images...
